In [1]:
import torch
import torch.nn as nn
import os
import cv2
import numpy as np
import mediapipe as mp
from tqdm import tqdm
from pathlib import Path
# MediaPipe 초기화
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True)

# 레이블 매핑
label_map = {"N": 0, "BY": 1, "FY": 2, "SY": 3}

X_data = []
y_data = []

In [2]:
from pathlib import Path
import torch
import torch.nn as nn
import os
import cv2
import numpy as np
import mediapipe as mp
from tqdm import tqdm
from pathlib import Path
# MediaPipe 초기화
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True)

# 레이블 매핑
label_map = {"N": 0, "BY": 1, "FY": 2, "SY": 3}

X_data = []
y_data = []

base_dir = "./Training/TS"
print(os.listdir("./Training/TS"))
# 경로 존재 여부 확인
if not os.path.exists(base_dir) or not os.listdir(base_dir):
    print(f"[경고] 경로가 존재하지 않거나 폴더가 비어 있습니다: {base_dir}")
else:
    print(f"'{base_dir}' 안에 있는 폴더 목록:")
    for folder in os.listdir(base_dir):
        path = os.path.join(base_dir, folder)
        if os.path.isdir(path):
            print("📁", folder)

['BY', 'FY', 'N', 'SY']
'./Training/TS' 안에 있는 폴더 목록:
📁 BY
📁 FY
📁 N
📁 SY


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from tqdm import tqdm
import re

# MediaPipe pose 초기화
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True)

# 라벨 매핑
label_map = {"N": 0, "BY": 1, "FY": 2, "SY": 3}

# 경로
base_dir = "./Validation/TS/"

# 데이터 리스트 초기화
X_data = []
y_data = []
seq_names = []

# 통계
cnt_false = 0
cnt_folder = 0

# 정렬 키
def numerical_sort_key(filename):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', filename)]

# 폴더 순회
for label_name in os.listdir(base_dir):
    label_path = os.path.join(base_dir, label_name)

    if not os.path.isdir(label_path):
        continue

    label_idx = label_map.get(label_name)
    if label_idx is None:
        continue

    for seq_folder in tqdm(os.listdir(label_path), desc=f"Processing {label_name}"):
        seq_path = os.path.join(label_path, seq_folder)
        if not os.path.isdir(seq_path):
            continue

        pose_sequence = []
        image_files = sorted(os.listdir(seq_path), key=numerical_sort_key)

        for img_name in image_files:
            img_path = os.path.join(seq_path, img_name)
            image = cv2.imread(img_path)

            if image is None:
                continue

            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            result = pose.process(image_rgb)

            if result.pose_landmarks:
                landmarks = result.pose_landmarks.landmark
                pose_vec = [coord for lm in landmarks for coord in (lm.x, lm.y, lm.z)]
                pose_sequence.append(pose_vec)
            else:
                pose_sequence.append([0.0] * 99)
                cnt_false += 1

        if len(pose_sequence) == 10:
            X_data.append(pose_sequence)
            y_data.append(label_idx)
            seq_names.append(seq_folder)
        else:
            cnt_folder += 1

# numpy 배열로 변환
X_data = np.array(X_data)  # (N, 10, 99)
y_data = np.array(y_data)
seq_names = np.array(seq_names)

print("데이터 처리 완료")
print("X shape:", X_data.shape)
print("y shape:", y_data.shape)
print("관절 못 찾은 이미지 수:", cnt_false)
print("10장 미만 폴더 수:", cnt_folder)

# CSV로 저장
if len(X_data) == 0:
    print("X_data가 비어 있습니다! 데이터 생성 확인 필요.")
else:
    X_flat = X_data.reshape(len(X_data), -1)  # (N, 990)
    feature_columns = [f"feat_{i}" for i in range(X_flat.shape[1])]

    df_all = pd.DataFrame(X_flat, columns=feature_columns)
    df_all.insert(0, "label", y_data)
    df_all.insert(0, "seq_name", seq_names)

    df_all.to_csv("fall_dataset_all_val.csv", index=False)
    print("CSV 저장 완료! shape:", df_all.shape)


Processing BY:   1%|          | 7/576 [00:06<08:59,  1.05it/s]


KeyboardInterrupt: 

In [9]:
import pandas as pd

# CSV 파일 읽기
df = pd.read_csv("fall_dataset_merged.csv")

# 결과 저장 리스트
rows_with_zeros = []

# 각 행마다 0의 개수 세기
for idx, row in df.iterrows():
    seq_name = row["seq_name"]
    # label, seq_name을 제외한 나머지 feature 값만 검사
    feature_values = row.drop(labels=["seq_name", "label"]).values
    zero_count = (feature_values == 0).sum()

    if zero_count > 0:
        rows_with_zeros.append((seq_name, zero_count))

# 결과 출력
print("📊 0값을 포함한 시퀀스 개수:", len(rows_with_zeros))
sv = 0
for seq_name, zero_count in rows_with_zeros:
    sv += zero_count
print(f"{sv} 0의 개수")

📊 0값을 포함한 시퀀스 개수: 11905
4059594 0의 개수


In [47]:
from mediapipe import solutions
import cv2

pose = solutions.pose.Pose(static_image_mode=True)
# img = cv2.imread("Training/TS/BY/01126_O_E_BY_C8/01126_O_E_BY_C8_I008.jpg")
img = cv2.imread("temp.jpg")
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 여러 번 시도
for i in range(5):
    result = pose.process(rgb)
    print(f"Try {i+1}: {'검출됨' if result.pose_landmarks else '❌ 검출 안됨'}")

Try 1: 검출됨
Try 2: 검출됨
Try 3: 검출됨
Try 4: 검출됨
Try 5: 검출됨


In [33]:
os.getcwd()  # 현재 작업 디렉토리 확인

'c:\\workspace\\project_3rd\\Datasets'

In [ ]:
import cv2
import os

# 이미지 경로 및 bbox 데이터
img_path = r"C:\workspace\project_3rd\Datasets\Validation\TS\N\02514_H_A_N_C1\02514_H_A_N_C1_I001.jpg"
bbox_location = "1667.9202880859375, 955.2904052734375, 2011.1834716796875, 1733.1229248046875"

# 이미지 읽기
image = cv2.imread(img_path)

if image is None:
    print("❌ 이미지 로드 실패!")
else:
    # bbox 좌표 파싱
    x_min, y_min, x_max, y_max = map(lambda v: int(float(v)), bbox_location.split(','))

    # 경계값이 이미지 크기를 넘지 않도록 클리핑
    h, w, _ = image.shape
    x_min = max(0, min(w - 1, x_min))
    x_max = max(0, min(w, x_max))
    y_min = max(0, min(h - 1, y_min))
    y_max = max(0, min(h, y_max))

    # 이미지 자르기
    cropped_img = image[y_min:y_max, x_min:x_max]

    # 저장 경로
    save_path = r"C:\workspace\project_3rd\cropped_bbox.jpg"
    cv2.imwrite(save_path, cropped_img)

    print(f"✅ 자른 이미지 저장 완료: {save_path}")

✅ 자른 이미지 저장 완료: C:\workspace\project_3rd\Datasets\Validation\TS\N\02514_H_A_N_C1\cropped_bbox.jpg


In [51]:
import os
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from tqdm import tqdm
import re
import json

# MediaPipe pose 초기화
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True)

# 라벨 매핑
label_map = {"N": 0, "BY": 1, "FY": 2, "SY": 3}

# 경로
base_dir = "./Validation/TS/"

# 데이터 리스트 초기화
X_data = []
y_data = []
seq_names = []

# 통계
cnt_false = 0
cnt_folder = 0

# 정렬 키
def numerical_sort_key(filename):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', filename)]


# 폴더 순회
for label_name in os.listdir(base_dir):
    label_path = os.path.join(base_dir, label_name)

    if not os.path.isdir(label_path):
        continue
    
    label_idx = label_map.get(label_name)
    if label_idx is None:
        continue

    for seq_folder in tqdm(os.listdir(label_path), desc=f"Processing {label_name}"):
        seq_path = os.path.join(label_path, seq_folder)
        if not os.path.isdir(seq_path):
            continue

        pose_sequence = []
        image_files = sorted(os.listdir(seq_path), key=numerical_sort_key)

        for img_name in image_files:
            img_path = os.path.join(seq_path, img_name)
            image = cv2.imread(img_path)

            if image is None:
                continue

            # bbox 정보 로드
            box_path = os.path.join("Validation/LABEL", label_name, seq_folder, img_name.replace(".jpg", ".json"))
            if not os.path.exists(box_path):
                continue  # bbox 정보 없으면 skip

            with open(box_path, "r") as f:
                box_data = json.load(f)

            bbox_str = box_data.get("bboxdata", {}).get("bbox_location")
            if not bbox_str:
                continue

            try:
                x_min, y_min, x_max, y_max = map(float, bbox_str.split(","))
                x_min, y_min, x_max, y_max = map(int, [x_min, y_min, x_max, y_max])
            except:
                continue

            # 이미지 자르기
            cropped = image[y_min:y_max, x_min:x_max]
            if cropped.size == 0:
                continue

            image_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
            result = pose.process(image_rgb)

            if result.pose_landmarks:
                landmarks = result.pose_landmarks.landmark
                pose_vec = []
                for lm in landmarks:
                    # 원본 이미지 기준 좌표로 복원
                    x_img = x_min + lm.x * (x_max - x_min)
                    y_img = y_min + lm.y * (y_max - y_min)
                    z = lm.z * (x_max - x_min)  # z도 scale 맞춰줌
                    pose_vec.extend([x_img, y_img, z])
                pose_sequence.append(pose_vec)
            else:
                pose_sequence.append([0.0] * 99)
                cnt_false += 1

        if len(pose_sequence) == 10:
            X_data.append(pose_sequence)
            y_data.append(label_idx)
            seq_names.append(seq_folder)
        else:
            cnt_folder += 1

# numpy 배열로 변환
X_data = np.array(X_data)  # (N, 10, 99)
y_data = np.array(y_data)
seq_names = np.array(seq_names)

print("✅ 데이터 처리 완료")
print("X shape:", X_data.shape)
print("y shape:", y_data.shape)
print("관절 못 찾은 이미지 수:", cnt_false)
print("10장 미만 폴더 수:", cnt_folder)

# CSV로 저장
if len(X_data) == 0:
    print("❌ X_data가 비어 있습니다! 데이터 생성 확인 필요.")
else:
    X_flat = X_data.reshape(len(X_data), -1)  # (N, 990)
    feature_columns = [f"feat_{i}" for i in range(X_flat.shape[1])]

    df_all = pd.DataFrame(X_flat, columns=feature_columns)
    df_all.insert(0, "label", y_data)
    df_all.insert(0, "seq_name", seq_names)

    df_all.to_csv("fall_dataset_all_val_bbox.csv", index=False)
    print("✅ CSV 저장 완료! shape:", df_all.shape)

Processing SY: 100%|██████████| 352/352 [03:21<00:00,  1.75it/s]


✅ 데이터 처리 완료
X shape: (2009, 10, 99)
y shape: (2009,)
관절 못 찾은 이미지 수: 5414
10장 미만 폴더 수: 263
✅ CSV 저장 완료! shape: (2009, 992)


In [52]:
import os
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from tqdm import tqdm
import re
import json

# MediaPipe pose 초기화
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True)

# 라벨 매핑
label_map = {"N": 0, "BY": 1, "FY": 2, "SY": 3}

# 경로
base_dir = "./Training/TS/"

# 데이터 리스트 초기화
X_data = []
y_data = []
seq_names = []

# 통계
cnt_false = 0
cnt_folder = 0

# 정렬 키
def numerical_sort_key(filename):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', filename)]


# 폴더 순회
for label_name in os.listdir(base_dir):
    label_path = os.path.join(base_dir, label_name)

    if not os.path.isdir(label_path):
        continue
    
    label_idx = label_map.get(label_name)
    if label_idx is None:
        continue

    for seq_folder in tqdm(os.listdir(label_path), desc=f"Processing {label_name}"):
        seq_path = os.path.join(label_path, seq_folder)
        if not os.path.isdir(seq_path):
            continue

        pose_sequence = []
        image_files = sorted(os.listdir(seq_path), key=numerical_sort_key)

        for img_name in image_files:
            img_path = os.path.join(seq_path, img_name)
            image = cv2.imread(img_path)

            if image is None:
                continue

            # bbox 정보 로드
            box_path = os.path.join("Training/LABEL", label_name, seq_folder, img_name.replace(".jpg", ".json"))
            if not os.path.exists(box_path):
                continue  # bbox 정보 없으면 skip

            with open(box_path, "r") as f:
                box_data = json.load(f)

            bbox_str = box_data.get("bboxdata", {}).get("bbox_location")
            if not bbox_str:
                continue

            try:
                x_min, y_min, x_max, y_max = map(float, bbox_str.split(","))
                x_min, y_min, x_max, y_max = map(int, [x_min, y_min, x_max, y_max])
            except:
                continue

            # 이미지 자르기
            cropped = image[y_min:y_max, x_min:x_max]
            if cropped.size == 0:
                continue

            image_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
            result = pose.process(image_rgb)

            if result.pose_landmarks:
                landmarks = result.pose_landmarks.landmark
                pose_vec = []
                for lm in landmarks:
                    # 원본 이미지 기준 좌표로 복원
                    x_img = x_min + lm.x * (x_max - x_min)
                    y_img = y_min + lm.y * (y_max - y_min)
                    z = lm.z * (x_max - x_min)  # z도 scale 맞춰줌
                    pose_vec.extend([x_img, y_img, z])
                pose_sequence.append(pose_vec)
            else:
                pose_sequence.append([0.0] * 99)
                cnt_false += 1

        if len(pose_sequence) == 10:
            X_data.append(pose_sequence)
            y_data.append(label_idx)
            seq_names.append(seq_folder)
        else:
            cnt_folder += 1

# numpy 배열로 변환
X_data = np.array(X_data)  # (N, 10, 99)
y_data = np.array(y_data)
seq_names = np.array(seq_names)

print("✅ 데이터 처리 완료")
print("X shape:", X_data.shape)
print("y shape:", y_data.shape)
print("관절 못 찾은 이미지 수:", cnt_false)
print("10장 미만 폴더 수:", cnt_folder)

# CSV로 저장
if len(X_data) == 0:
    print("❌ X_data가 비어 있습니다! 데이터 생성 확인 필요.")
else:
    X_flat = X_data.reshape(len(X_data), -1)  # (N, 990)
    feature_columns = [f"feat_{i}" for i in range(X_flat.shape[1])]

    df_all = pd.DataFrame(X_flat, columns=feature_columns)
    df_all.insert(0, "label", y_data)
    df_all.insert(0, "seq_name", seq_names)

    df_all.to_csv("fall_dataset_all_training_bbox.csv", index=False)
    print("✅ CSV 저장 완료! shape:", df_all.shape)

Processing SY: 100%|██████████| 1872/1872 [19:31<00:00,  1.60it/s]


✅ 데이터 처리 완료
X shape: (12194, 10, 99)
y shape: (12194,)
관절 못 찾은 이미지 수: 27852
10장 미만 폴더 수: 278
✅ CSV 저장 완료! shape: (12194, 992)


In [8]:
import pandas as pd

# 파일 경로
file_1 = r"C:/workspace/project_3rd/Datasets/fall_dataset_all_val_bbox_1000.csv"
file_2 = r"C:/workspace/project_3rd/Datasets/fall_dataset_all_training_bbox.csv"
output_file = r"C:/workspace/project_3rd/Datasets/fall_dataset_merged.csv"

# CSV 불러오기
df1 = pd.read_csv(file_1)
df2 = pd.read_csv(file_2)

# 이어붙이기
merged_df = pd.concat([df1, df2], ignore_index=True)

# 저장
merged_df.to_csv(output_file, index=False)

print(f"병합 완료! 저장된 파일: {output_file}")


병합 완료! 저장된 파일: C:/workspace/project_3rd/Datasets/fall_dataset_merged.csv


## 결측치 채우기

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# --- 설정값 ---
INPUT_CSV_PATH = "fall_dataset_merged.csv"  # 원본 CSV 파일 경로
OUTPUT_CSV_PATH = "fall_dataset_interpolated_descriptive3.csv" # 저장할 CSV 파일 경로

NUM_FRAMES = 10
NUM_LANDMARKS = 33
NUM_COORDS = 3 # x, y, z
FEATURES_PER_FRAME = NUM_LANDMARKS * NUM_COORDS # 99
TOTAL_FEATURES = NUM_FRAMES * FEATURES_PER_FRAME # 990

MIN_DETECTED_FRAMES_PER_LANDMARK = 4 # 랜드마크당 최소 감지 프레임 수

# --- 1. 데이터 로드 ---
print(f"🔄 '{INPUT_CSV_PATH}' 파일 로딩 중...")
try:
    df_raw = pd.read_csv(INPUT_CSV_PATH)
    print(f"✅ 파일 로드 완료. Shape: {df_raw.shape}")
except FileNotFoundError:
    print(f"❌ 에러: '{INPUT_CSV_PATH}' 파일을 찾을 수 없습니다. 경로를 확인하세요.")
    exit()
except Exception as e:
    print(f"❌ 에러: CSV 파일 로딩 중 오류 발생 - {e}")
    exit()

🔄 'fall_dataset_merged.csv' 파일 로딩 중...
✅ 파일 로드 완료. Shape: (16619, 992)


In [2]:
# --- 2. 데이터 준비 ---
# seq_name과 label 컬럼 분리
if 'seq_name' not in df_raw.columns or 'label' not in df_raw.columns:
    print("❌ 에러: 입력 CSV에 'seq_name' 또는 'label' 컬럼이 없습니다.")
    exit()

seq_names_raw = df_raw['seq_name'].values
y_data_raw = df_raw['label'].values

# Feature 컬럼만 선택 (seq_name, label 제외)
# 컬럼 이름이 feat_0 ~ feat_989 라고 가정
expected_feature_cols = [f"feat_{i}" for i in range(TOTAL_FEATURES)]
actual_feature_cols = [col for col in df_raw.columns if col.startswith('feat_')]

if len(actual_feature_cols) != TOTAL_FEATURES:
     # 만약 컬럼 이름이 다르다면, seq_name과 label을 제외한 나머지 컬럼을 feature로 간주
     print(f"⚠️ 경고: Feature 컬럼 이름이 'feat_0' ~ 'feat_{TOTAL_FEATURES-1}' 형식이 아닙니다.")
     print("    'seq_name'과 'label'을 제외한 나머지 컬럼을 Feature로 사용합니다.")
     feature_columns = [col for col in df_raw.columns if col not in ['seq_name', 'label']]
     if len(feature_columns) != TOTAL_FEATURES:
         print(f"❌ 에러: Feature 컬럼 수가 {TOTAL_FEATURES}개가 아닙니다 ({len(feature_columns)}개 발견).")
         exit()
     X_flat_raw = df_raw[feature_columns].values
else:
    X_flat_raw = df_raw[expected_feature_cols].values


# 3D 형태로 변환: (N, 990) -> (N, 10, 99)
num_sequences = X_flat_raw.shape[0]
try:
    X_data_raw = X_flat_raw.reshape(num_sequences, NUM_FRAMES, FEATURES_PER_FRAME)
    print(f"✅ 데이터를 (N, Frames, Features) 형태로 변환 완료: {X_data_raw.shape}")
except ValueError as e:
    print(f"❌ 에러: 데이터를 ({num_sequences}, {NUM_FRAMES}, {FEATURES_PER_FRAME}) 형태로 변환할 수 없습니다.")
    print(f"    원본 데이터의 Feature 수가 {TOTAL_FEATURES}개인지 확인하세요. 오류: {e}")
    exit()


✅ 데이터를 (N, Frames, Features) 형태로 변환 완료: (16619, 10, 99)


In [3]:
# --- 3. 필터링 ---
cnt_removed_few_landmarks = 0
indices_to_keep = []

print(f"\n🔍 시퀀스 필터링 시작 (랜드마크당 최소 {MIN_DETECTED_FRAMES_PER_LANDMARK} 프레임 감지 필요)")

for i in tqdm(range(X_data_raw.shape[0]), desc="Filtering Sequences"):
    sequence = X_data_raw[i] # (10, 99)
    is_valid_sequence = True

    # 각 랜드마크별로 감지된 프레임 수 계산
    for landmark_idx in range(NUM_LANDMARKS):
        coord_start_idx = landmark_idx * NUM_COORDS
        # x 좌표가 0이 아니면 해당 프레임에서 랜드마크가 감지된 것으로 간주
        detected_frames = np.sum(sequence[:, coord_start_idx] != 0.0)

        if detected_frames < MIN_DETECTED_FRAMES_PER_LANDMARK:
            is_valid_sequence = False
            break # 하나라도 조건 미달이면 더 볼 필요 없음

    if is_valid_sequence:
        indices_to_keep.append(i)
    else:
        cnt_removed_few_landmarks += 1

# 필터링된 데이터 선택
X_data_filtered = X_data_raw[indices_to_keep]
y_data_filtered = y_data_raw[indices_to_keep]
seq_names_filtered = seq_names_raw[indices_to_keep]

print(f"✅ 필터링 완료: {len(indices_to_keep)} / {X_data_raw.shape[0]} 시퀀스 유지")
print(f"🚫 제거된 시퀀스 수 (랜드마크 감지 부족): {cnt_removed_few_landmarks}")

if X_data_filtered.size == 0:
    print("❌ 필터링 후 남은 데이터가 없습니다!")
    exit()



🔍 시퀀스 필터링 시작 (랜드마크당 최소 4 프레임 감지 필요)


Filtering Sequences: 100%|██████████| 16619/16619 [00:02<00:00, 8263.23it/s]


✅ 필터링 완료: 15443 / 16619 시퀀스 유지
🚫 제거된 시퀀스 수 (랜드마크 감지 부족): 1176


In [4]:
import numpy as np
import pandas as pd
from tqdm import tqdm

# --- 4. 보간 (Pandas 스플라인 사용) ---
print("\n✨ 결측치 보간 시작 (Pandas 스플라인 보간)")

# 보간된 데이터를 저장할 배열 초기화
X_data_interpolated = np.zeros_like(X_data_filtered)

# 각 시퀀스에 대해 보간 수행
for i in tqdm(range(X_data_filtered.shape[0]), desc="Interpolating Data"):
    sequence = X_data_filtered[i].copy()  # 현재 시퀀스 복사 (10, 99)

    # 0을 NaN으로 변경 (Pandas interpolate가 NaN을 인식하도록)
    sequence[sequence == 0.0] = np.nan

    # Pandas DataFrame으로 변환 (프레임이 행, 특징이 열)
    df_seq = pd.DataFrame(sequence)

    try:
        df_interpolated = df_seq.interpolate(
            method="spline", order=3, axis=0, limit_direction="both"
        )

        # 보간 후에도 남아있는 NaN 값은 0으로 채움 (예: 유효값이 부족했던 열)
        df_interpolated = df_interpolated.fillna(0.0)

        # 보간된 데이터를 NumPy 배열로 변환하여 저장
        X_data_interpolated[i] = df_interpolated.values

    except Exception as e:
        print(f"\n⚠️ 시퀀스 {i} 보간 중 오류 발생: {e}")
        print("    해당 시퀀스는 원본 (0 포함) 또는 부분 보간 상태로 남을 수 있습니다.")
        # 오류 발생 시 대처 방안 결정 (예: 원본 데이터 유지 또는 0으로 채우기)
        # 여기서는 오류 발생 시 변경 없이 (NaN 포함 가능) 또는 부분 보간된 df_seq 사용 시도
        X_data_interpolated[i] = df_seq.fillna(0.0).values  # 임시 방편: NaN만 0으로

print("✅ 보간 완료")
print("보간 후 X shape (3D):", X_data_interpolated.shape)


✨ 결측치 보간 시작 (Pandas 스플라인 보간)


Interpolating Data:   0%|          | 0/15443 [00:00<?, ?it/s]

c:\Users\hwan7\.conda\envs\env312_cuda124_torch260\Lib\site-packages\pandas\core\missing.py:604: UserWarning: 
The maximal number of iterations maxit (set to 20 by the program)
allowed for finding a smoothing spline with fp=s has been reached: s
too small.
There is an approximation returned but the corresponding weighted sum
of squared residuals does not satisfy the condition abs(fp-s)/s < tol.
  terp = interpolate.UnivariateSpline(x, y, k=order, **kwargs)
Interpolating Data:   0%|          | 1/15443 [00:00<1:15:27,  3.41it/s]c:\Users\hwan7\.conda\envs\env312_cuda124_torch260\Lib\site-packages\pandas\core\missing.py:604: UserWarning: 
The maximal number of iterations maxit (set to 20 by the program)
allowed for finding a smoothing spline with fp=s has been reached: s
too small.
There is an approximation returned but the corresponding weighted sum
of squared residuals does not satisfy the condition abs(fp-s)/s < tol.
  terp = interpolate.UnivariateSpline(x, y, k=order, **kwargs)
c:\User

✅ 보간 완료
보간 후 X shape (3D): (15443, 10, 99)


In [5]:
# --- 5. 최종 데이터 준비 및 저장 ---
# CSV 저장을 위해 2D로 다시 펼치기: (N_filtered, 10, 99) -> (N_filtered, 990)
X_final_flat = X_data_interpolated.reshape(X_data_interpolated.shape[0], -1)

# 새로운 설명적인 컬럼 이름 생성
coords = ['x', 'y', 'z']
new_feature_columns = []
for f in range(NUM_FRAMES):
    for lm in range(NUM_LANDMARKS):
        for c in coords:
            new_feature_columns.append(f"frame{f}_lm{lm}_{c}")

if len(new_feature_columns) != TOTAL_FEATURES:
     print(f"❌ 에러: 생성된 새 컬럼명 개수({len(new_feature_columns)})가 예상 특징 수({TOTAL_FEATURES})와 다릅니다.")
     exit()

# 최종 DataFrame 생성
df_final = pd.DataFrame(X_final_flat, columns=new_feature_columns)
df_final.insert(0, "label", y_data_filtered)
df_final.insert(0, "seq_name", seq_names_filtered)

# CSV로 저장
df_final.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"\n✅ 최종 데이터 CSV 저장 완료!")
print(f"   경로: {OUTPUT_CSV_PATH}")
print(f"   Shape: {df_final.shape}")


✅ 최종 데이터 CSV 저장 완료!
   경로: fall_dataset_interpolated_descriptive3.csv
   Shape: (15443, 992)


In [6]:
import pandas as pd


def update_labels(file_path):
    # CSV 파일 읽기
    df = pd.read_csv(file_path)

    # "label" 열 값 변경
    label_mapping = {0: "N", 1: "BY", 2: "FY", 3: "SY"}
    df["label"] = df["label"].map(label_mapping)

    # 변경된 데이터프레임 반환
    return df


if __name__ == "__main__":
    # 파일 경로 지정
    file_path = "fall_dataset_interpolated_descriptive3.csv"

    # label 값 변경
    updated_df = update_labels(file_path)

    # 결과 출력 (또는 파일로 저장)
    print(updated_df.head())  # 데이터프레임 상위 5개 행 출력

    # 변경된 파일 저장 (선택 사항)
    updated_df.to_csv("updated_fall_dataset.csv", index=False)

          seq_name label  frame0_lm0_x  frame0_lm0_y  frame0_lm0_z  \
0  00050_H_A_BY_C1    BY   2410.048569    577.581797   -140.053708   
1  00050_H_A_BY_C2    BY   2118.185027    517.723994    -50.215824   
2  00050_H_A_BY_C3    BY   1842.920149    554.047858   -357.667368   
3  00050_H_A_BY_C5    BY   1851.576045    643.936327   -234.184700   
4  00050_H_A_BY_C7    BY   1837.924424    558.882133   -493.935251   

   frame0_lm1_x  frame0_lm1_y  frame0_lm1_z  frame0_lm2_x  frame0_lm2_y  ...  \
0   2403.246593    534.713993   -152.275960   2397.967543    532.292332  ...   
1   2123.738221    484.053879   -113.905829   2125.548048    483.446300  ...   
2   1851.724144    534.206271   -375.741830   1860.934230    533.159109  ...   
3   1846.634015    617.247477   -209.584366   1844.326815    615.777525  ...   
4   1848.445028    537.494044   -470.634569   1855.697013    536.932105  ...   

   frame9_lm29_z  frame9_lm30_x  frame9_lm30_y  frame9_lm30_z  frame9_lm31_x  \
0    1509.776373  

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm  # 진행 상황 표시를 위해 tqdm 추가

# --- 상수 정의 ---
INPUT_CSV_PATH = "updated_fall_dataset.csv"
OUTPUT_CSV_PATH = "normalized_fall_dataset.csv"
NUM_FRAMES = 10
NUM_LANDMARKS = 33  # MediaPipe Pose landmarks (0-32)

# 정규화를 위한 기준 랜드마크 인덱스 (MediaPipe Pose 기준)
LM_IDX_LEFT_HIP = 23
LM_IDX_RIGHT_HIP = 24
LM_IDX_LEFT_SHOULDER = 11
LM_IDX_RIGHT_SHOULDER = 12

# 작은 값 (Division by zero 방지)
EPSILON = 1e-6

# --- 데이터 로드 ---
try:
    df = pd.read_csv(INPUT_CSV_PATH)
    print(f"✅ 원본 데이터 로드 완료: {INPUT_CSV_PATH} (총 {len(df)}개 시퀀스)")
except FileNotFoundError:
    print(f"❌ 오류: 파일을 찾을 수 없습니다 - {INPUT_CSV_PATH}")
    exit()

# --- 정규화 수행 ---
normalized_data_list = []

print("🚀 데이터 정규화 시작...")
# tqdm을 사용하여 진행률 표시
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Processing Sequences"):
    normalized_row_dict = {"seq_name": row["seq_name"], "label": row["label"]}
    all_frames_normalized_coords = []

    for f in range(NUM_FRAMES):
        # 현재 프레임의 모든 랜드마크 좌표 추출 (NUM_LANDMARKS x 3 형태)
        frame_coords = np.zeros((NUM_LANDMARKS, 3))
        try:
            for lm in range(NUM_LANDMARKS):
                frame_coords[lm, 0] = row[f"frame{f}_lm{lm}_x"]
                frame_coords[lm, 1] = row[f"frame{f}_lm{lm}_y"]
                frame_coords[lm, 2] = row[f"frame{f}_lm{lm}_z"]
        except KeyError as e:
            print(
                f"\n❌ 오류: 컬럼 누락 - {e}. 시퀀스 {row['seq_name']}, 프레임 {f} 처리 중 문제 발생."
            )
            # 오류 발생 시 해당 시퀀스는 건너뛰거나 다른 처리 가능
            # 여기서는 예시로 None을 추가하고 다음 프레임/시퀀스로 넘어감
            frame_coords = None  # 오류 표시
            break  # 현재 프레임 처리 중단

        if frame_coords is None:
            all_frames_normalized_coords = None  # 시퀀스 전체를 잘못된 것으로 표시
            break  # 현재 시퀀스 처리 중단

        # 1. 골반 중심 계산 (Translation 기준점)
        left_hip = frame_coords[LM_IDX_LEFT_HIP]
        right_hip = frame_coords[LM_IDX_RIGHT_HIP]
        hip_center = (left_hip + right_hip) / 2.0

        # 2. 어깨 너비 계산 (Scale 기준 거리)
        left_shoulder = frame_coords[LM_IDX_LEFT_SHOULDER]
        right_shoulder = frame_coords[LM_IDX_RIGHT_SHOULDER]
        # np.linalg.norm : 두 점 사이의 유클리드 거리 계산
        shoulder_width = np.linalg.norm(left_shoulder - right_shoulder) + EPSILON

        # 3. 정규화 수행: (좌표 - 골반중심) / 어깨너비
        normalized_coords = (frame_coords - hip_center) / shoulder_width

        # 정규화된 좌표를 원래 컬럼명 형식으로 저장
        for lm in range(NUM_LANDMARKS):
            normalized_row_dict[f"frame{f}_lm{lm}_x"] = normalized_coords[lm, 0]
            normalized_row_dict[f"frame{f}_lm{lm}_y"] = normalized_coords[lm, 1]
            normalized_row_dict[f"frame{f}_lm{lm}_z"] = normalized_coords[lm, 2]

    # 유효하게 처리된 시퀀스만 결과 리스트에 추가
    if all_frames_normalized_coords is not None:
        normalized_data_list.append(normalized_row_dict)


# --- 정규화된 데이터프레임 생성 및 저장 ---
if normalized_data_list:  # 처리된 데이터가 있을 경우에만 저장
    normalized_df = pd.DataFrame(normalized_data_list)
    normalized_df.to_csv(OUTPUT_CSV_PATH, index=False)
    print(
        f"\n✅ 데이터 정규화 완료. {len(normalized_df)}개 시퀀스가 저장되었습니다: {OUTPUT_CSV_PATH}"
    )
else:
    print("\n⚠️ 처리된 유효한 시퀀스가 없어 파일을 저장하지 않았습니다.")

✅ 원본 데이터 로드 완료: updated_fall_dataset.csv (총 15443개 시퀀스)
🚀 데이터 정규화 시작...


Processing Sequences: 100%|██████████| 15443/15443 [00:37<00:00, 411.25it/s]



✅ 데이터 정규화 완료. 15443개 시퀀스가 저장되었습니다: normalized_fall_dataset.csv
